# Data Transformation (Bronze → Silver)

This notebook reads `transformation_rules` metadata to dynamically apply column-level transformations from bronze tables into the silver layer. Each rule specifies a source column, target column, SQL expression, and target data type — all driven by data, not hardcoded.

In [0]:
import uuid
from datetime import datetime
from pyspark.sql.functions import current_timestamp

CATALOG = "metadata_framework"
META_SCHEMA = f"{CATALOG}.bronze"
SILVER_SCHEMA = f"{CATALOG}.silver"

def get_transformation_rules(table_id):
    """Fetch all active transformation rules for a given table_id, ordered by execution_order."""
    return (spark.read.table(f"{META_SCHEMA}.transformation_rules")
            .where(f"table_id = '{table_id}' AND is_active = true")
            .orderBy("execution_order")
            .collect())

def get_table_config(table_id):
    """Fetch table configuration for a given table_id."""
    return (spark.read.table(f"{META_SCHEMA}.table_config")
            .where(f"table_id = '{table_id}' AND is_active = true")
            .collect()[0])

def log_pipeline_run(run_id, table_id, status, layer, records_read=0, records_written=0, records_rejected=0, error_message=None):
    """Insert a record into pipeline_run_log."""
    log_df = spark.createDataFrame([(
        run_id, table_id, status, layer,
        records_read, records_written, records_rejected,
        error_message,
        datetime.now(),
        datetime.now() if status in ("success", "failed") else None
    )], schema="run_id STRING, table_id STRING, run_status STRING, run_layer STRING, records_read BIGINT, records_written BIGINT, records_rejected BIGINT, error_message STRING, started_at TIMESTAMP, completed_at TIMESTAMP")
    log_df.write.mode("append").saveAsTable(f"{META_SCHEMA}.pipeline_run_log")

print("Transformation utilities loaded.")

In [0]:
def transform_table(table_id):
    """
    Transform a bronze table into a silver table using metadata-driven transformation rules.
    Builds a SELECT statement dynamically from the transformation_rules metadata.
    """
    run_id = str(uuid.uuid4())
    config = get_table_config(table_id)
    
    bronze_table = f"{CATALOG}.bronze.{config.target_table_name}"
    silver_table = f"{SILVER_SCHEMA}.{config.target_table_name.replace('raw_', '')}"
    
    rules = get_transformation_rules(table_id)
    if not rules:
        print(f"No active transformation rules for {table_id}, skipping.")
        return False
    
    print(f"\n{'='*60}")
    print(f"Transforming: {bronze_table} → {silver_table}")
    print(f"Rules: {len(rules)} transformation(s)")
    
    log_pipeline_run(run_id, table_id, "started", "silver")
    
    try:
        # Build SELECT clause from transformation rules
        select_exprs = []
        for rule in rules:
            # Each rule: transformation_sql AS target_column
            select_exprs.append(f"{rule.transformation_sql} AS {rule.target_column}")
        
        # Add metadata columns
        select_exprs.append("_ingested_at AS bronze_ingested_at")
        select_exprs.append("current_timestamp() AS silver_transformed_at")
        
        select_clause = ",\n      ".join(select_exprs)
        
        # Build and execute the CREATE OR REPLACE TABLE statement
        create_sql = f"""
        CREATE OR REPLACE TABLE {silver_table}
        AS
        SELECT
      {select_clause}
        FROM {bronze_table}
        """
        
        print(f"Generated SQL:\n{create_sql}")
        spark.sql(create_sql)
        
        records_read = spark.read.table(bronze_table).count()
        records_written = spark.read.table(silver_table).count()
        
        print(f"✅ Success: {records_read} → {records_written} records transformed")
        log_pipeline_run(run_id, table_id, "success", "silver", records_read, records_written)
        return True
        
    except Exception as e:
        error_msg = str(e)[:500]
        print(f"❌ Failed: {error_msg}")
        log_pipeline_run(run_id, table_id, "failed", "silver", 0, 0, 0, error_msg)
        return False

print("Transform function defined.")

In [0]:
# ============================================================================
# Execute transformations for all active tables that have transformation rules
# ============================================================================
active_tables = (spark.read.table(f"{META_SCHEMA}.table_config")
                 .where("is_active = true")
                 .select("table_id")
                 .collect())

success_count = 0
failure_count = 0

for row in active_tables:
    result = transform_table(row.table_id)
    if result:
        success_count += 1
    else:
        failure_count += 1

print(f"\n{'='*60}")
print(f"Transformation Summary: {success_count} succeeded, {failure_count} failed")